# Data Cleaning and Preprocessing

Notebook for preprocessing: apply the preprocessing pipeline, compare before vs after, and persist the cleaned dataset.

## 1) Imports and setup

Load dependencies and configure project path for importing `src.preprocessing`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.preprocessing import run_preprocessing_pipeline
from src.parquet_io import write_dataframe_parquet

## 2) Load raw dataset

Read the raw weather CSV from `data/raw`.

In [2]:
raw_path = project_root / "data" / "raw" / "GlobalWeatherRepository.csv"
df_raw = pd.read_csv(raw_path)

print("Raw path:", raw_path)
print("Raw shape:", df_raw.shape)
display(df_raw.head(3))

Raw path: c:\Users\lucas\Desktop\pma-weather-forecasting\data\raw\GlobalWeatherRepository.csv
Raw shape: (133318, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55


## 3) Baseline profile (before preprocessing)

Capture nulls, descriptive stats, and shape before transformations.

In [3]:
nulls_before = df_raw.isna().sum().sort_values(ascending=False)
null_pct_before = ((nulls_before / len(df_raw)) * 100).round(2)

print("Shape before:", df_raw.shape)
display(nulls_before.head(10))
display(null_pct_before.head(10))
display(df_raw.describe(include="all").transpose().head(10))

Shape before: (133318, 41)


country                   0
location_name             0
latitude                  0
longitude                 0
timezone                  0
last_updated_epoch        0
last_updated              0
temperature_celsius       0
temperature_fahrenheit    0
condition_text            0
dtype: int64

country                   0.0
location_name             0.0
latitude                  0.0
longitude                 0.0
timezone                  0.0
last_updated_epoch        0.0
last_updated              0.0
temperature_celsius       0.0
temperature_fahrenheit    0.0
condition_text            0.0
dtype: float64

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,133318,211,Bulgaria,1583,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_name,133318,257,Kabul,686,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,133318.0,NaN,NaN,NaN,19.207034,24.417913,-41.3,4.050075,17.25,40.4,64.15
longitude,133318.0,NaN,NaN,NaN,21.966088,65.787537,-175.2,-6.8361,23.2361,50.40555,179.22
timezone,133318,199,Asia/Bangkok,2466,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_updated_epoch,133318.0,NaN,NaN,NaN,1745516451.281147,17119922.381598,1715849100.0,1730710800.0,1745570700.0,1760342400.0,1775197800.0
last_updated,133318,22275,2025-12-26 08:15,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperature_celsius,133318.0,NaN,NaN,NaN,21.316548,9.703496,-29.8,16.0,24.0,28.0,49.2
temperature_fahrenheit,133318.0,NaN,NaN,NaN,70.371554,17.466153,-21.6,60.8,75.2,82.4,120.6
condition_text,133318,49,Partly cloudy,38711,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4) Run preprocessing pipeline

Apply imputation, IQR clipping, MinMax normalization, and one-hot encoding.

In [4]:
df_clean, artifacts = run_preprocessing_pipeline(df_raw)

print("Shape after:", df_clean.shape)
print("Encoded columns count:", len(df_clean.columns))
print("Scaled columns count:", len(artifacts["minmax_columns"]))
print("Excluded from normalization:", artifacts["exclude_from_normalize"])
display(df_clean.head(3))

Shape after: (133318, 26975)
Encoded columns count: 26975
Scaled columns count: 27
Excluded from normalization: ['last_updated_epoch', 'latitude', 'longitude']


,latitude,longitude,last_updated_epoch,temperature_celsius,temperature_fahrenheit,wind_mph,wind_kph,wind_degree,pressure_mb,pressure_in,...,moonset_24:00,moonset_No moonset,moon_phase_First Quarter,moon_phase_Full Moon,moon_phase_Last Quarter,moon_phase_New Moon,moon_phase_Waning Crescent,moon_phase_Waning Gibbous,moon_phase_Waxing Crescent,moon_phase_Waxing Gibbous
0,34.52,69.18,1715849100,0.595833,0.594907,0.311224,0.3104,0.938719,0.43750,0.440217,...,False,False,False,False,False,False,False,False,False,True
1,41.33,19.82,1715849100,0.437500,0.437500,0.239796,0.2432,0.888579,0.43750,0.429348,...,False,False,False,False,False,False,False,False,False,True
2,36.76,3.05,1715849100,0.520833,0.520833,0.367347,0.3680,0.777159,0.40625,0.396739,...,False,False,False,False,False,False,False,False,False,True


## 5) Before vs after comparison

Compare missing values, shape, and outlier diagnostics.

In [5]:
nulls_after = df_clean.isna().sum().sort_values(ascending=False)

comparison = pd.DataFrame(
    {
        "nulls_before": nulls_before,
        "nulls_after": nulls_after.reindex(nulls_before.index, fill_value=0),
    }
).sort_values("nulls_before", ascending=False)

# Outlier count summary based on IQR bounds (before clipping)
outlier_counts = {}
for col, (low, high) in artifacts["iqr_bounds"].items():
    if col in df_raw.columns:
        outlier_counts[col] = int(((df_raw[col] < low) | (df_raw[col] > high)).sum())

outlier_counts = pd.Series(outlier_counts).sort_values(ascending=False)

print("Shape before -> after:", df_raw.shape, "->", df_clean.shape)
display(comparison.head(15))
display(outlier_counts.head(15))

Shape before -> after: (133318, 41) -> (133318, 26975)


,nulls_before,nulls_after
country,0,0
location_name,0,0
latitude,0,0
longitude,0,0
timezone,0,0
last_updated_epoch,0,0
last_updated,0,0
temperature_celsius,0,0
temperature_fahrenheit,0,0
condition_text,0,0


visibility_km                   28655
visibility_miles                28491
precip_mm                       24780
precip_in                       20692
air_quality_Sulphur_dioxide     18400
air_quality_PM10                14138
air_quality_Nitrogen_dioxide    13767
air_quality_gb-defra-index      12756
air_quality_Carbon_Monoxide     12075
air_quality_PM2.5               11021
longitude                       10253
air_quality_us-epa-index         9283
pressure_in                      5212
pressure_mb                      4082
gust_mph                         3635
dtype: int64

## 6) Save cleaned dataset

Persist the cleaned dataframe to Parquet under `data/processed/`. Saving uses `src.parquet_io.write_dataframe_parquet` (PyArrow directly) to avoid a known Jupyter issue with `df.to_parquet` (`ArrowKeyError: pandas.period already defined`).

In [6]:
output_path = project_root / "data" / "processed" / "cleaned_weather.parquet"

write_dataframe_parquet(df_clean, output_path, preserve_index=False)

print("Saved:", output_path)
print("File exists:", output_path.exists())

Saved: c:\Users\lucas\Desktop\pma-weather-forecasting\data\processed\cleaned_weather.parquet
File exists: True


## 7) Final notes

Key decisions in this pipeline:
- Missing values: median (numeric), mode (categorical).
- Outliers: IQR fences with clipping (`strategy='clip'`).
- Normalization: MinMax scaling with explicit exclusions (`last_updated_epoch`, `latitude`, `longitude`).
- Categorical variables: one-hot encoding via `pd.get_dummies`.

If cardinality growth becomes too high, the next iteration should evaluate target/ordinal encoding for selected columns.